# Perfis e `GRANT` por camada (etapa 6.2)

Concede o acesso de cada perfil conforme o [ADR 0016](../../docs/adr/0016-perfis-grupos-e-grants-por-camada.md) e **prova** o resultado lendo os privilégios gravados no catálogo. Fica registrado em `olist.quality.validacao_governanca`.

**Antes de rodar:** os 4 grupos precisam existir na conta (criados na interface, ver o passo a passo do diário da etapa): `olist_analista`, `olist_gestor`, `olist_dpo`, `olist_engenharia`. A célula 2 confere e para com uma mensagem clara se faltar algum.

| Perfil | Acesso concedido |
|---|---|
| `olist_analista`, `olist_gestor` | `SELECT` objeto a objeto na `gold` (exceto tabelas de apoio e log do pipeline) e nas tabelas `quality.validacao_*` |
| `olist_dpo` | `SELECT` em `bronze`, `silver`, `gold`, `quality` e `ops` |
| `olist_engenharia` | todos os privilégios no catálogo `olist` (é o grupo isento das máscaras) |

**Parâmetro `modo`:** `simulacao` (padrão) só mostra os comandos; `executar` aplica e verifica. Rode primeiro em `simulacao`. Pode rodar `executar` várias vezes: `GRANT` repetido não muda nada.

In [0]:
# O QUE FAZ: define o catálogo, o parâmetro de modo e os nomes dos grupos; confere que os 4 grupos existem na conta.
#
# FUNÇÕES DECLARADAS:
# - nenhuma (só configuração).
#
# FUNÇÕES NATIVAS E SQL:
# - dbutils.widgets.dropdown(nome, padrão, opções, rótulo): cria o parâmetro `modo` no topo do notebook.
# - dbutils.widgets.get(nome): lê o valor escolhido.
# - SHOW GROUPS: lista os grupos de conta visíveis ao usuário; grant só vale para grupo de conta.
# - raise RuntimeError(texto): interrompe a execução com a mensagem.
from pyspark.sql import functions as F

CATALOG = "olist"
dbutils.widgets.dropdown("modo", "simulacao", ["simulacao", "executar"], "Modo")
MODO = dbutils.widgets.get("modo")

GRUPOS = {
    "analista": "olist_analista",
    "gestor": "olist_gestor",
    "dpo": "olist_dpo",
    "engenharia": "olist_engenharia",
}
CONSUMIDORES = ["analista", "gestor"]
SCHEMAS_DPO = ["bronze", "silver", "gold", "quality", "ops"]
VALIDACOES = ["validacao_resultados", "validacao_gold", "validacao_live", "validacao_governanca"]

existentes = {r[0] for r in spark.sql("SHOW GROUPS").collect()}
faltando = [g for g in GRUPOS.values() if g not in existentes]
if faltando:
    raise RuntimeError(f"Crie primeiro os grupos de conta que faltam: {faltando}. Grupos vistos: {sorted(existentes)}")
print(f"modo = {MODO}; grupos conferidos: {list(GRUPOS.values())}")

In [0]:
# O QUE FAZ: cria (se não existir) a tabela quality.validacao_governanca e declara a função que grava resultados.
#
# FUNÇÕES DECLARADAS:
# - registrar(etapa, linhas, apagar_antes): apaga as linhas anteriores da etapa (ou dos perfis) e grava as novas em
#   quality.validacao_governanca. Rodar de novo troca o resultado, não duplica.
#
# FUNÇÕES NATIVAS E SQL:
# - CREATE TABLE IF NOT EXISTS ... COMMENT: cria a tabela de resultados uma vez, já com comentário em tabela e colunas.
# - DELETE FROM t WHERE ...: remove as linhas antigas (Delta). spark.createDataFrame(linhas, "schema DDL"): cria o
#   DataFrame a partir de tuplas. F.lit(x): coluna com valor fixo. F.current_timestamp(): hora da execução.
# - df.write.mode("append").saveAsTable(t): acrescenta as linhas à tabela.

VALIDACAO = f"{CATALOG}.quality.validacao_governanca"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDACAO} (
  etapa STRING COMMENT 'Etapa da Fase 6 que gerou a verificação (6.2-grants, 6.2-mascaras, 6.2-perfil, ...).',
  perfil STRING COMMENT 'Perfil avaliado (analista, gestor, dpo, engenharia) ou geral quando vale para todos.',
  verificacao STRING COMMENT 'O que foi verificado.',
  esperado STRING COMMENT 'Valor esperado pela regra.',
  obtido STRING COMMENT 'Valor encontrado no ambiente.',
  resultado STRING COMMENT 'OK, REPROVADA ou REGISTRO (só guarda um valor de referência).',
  executado_em TIMESTAMP COMMENT 'Momento da execução.'
)
COMMENT 'Resultado das verificações de governança e LGPD (grants, máscaras, filtros de linha, exclusão de cliente). Nenhuma linha guarda dado pessoal.'
""")


def registrar(etapa: str, linhas: list, apagar_antes: str = "etapa") -> None:
    """Grava as verificações. linhas = [(perfil, verificacao, esperado, obtido, resultado)]."""
    if apagar_antes == "etapa":
        spark.sql(f"DELETE FROM {VALIDACAO} WHERE etapa = '{etapa}'")
    else:
        perfis = sorted({l[0] for l in linhas})
        lista = ", ".join(f"'{p}'" for p in perfis)
        spark.sql(f"DELETE FROM {VALIDACAO} WHERE etapa = '{etapa}' AND perfil IN ({lista})")
    df = (spark.createDataFrame(linhas, "perfil string, verificacao string, esperado string, obtido string, resultado string")
          .withColumn("etapa", F.lit(etapa))
          .withColumn("executado_em", F.current_timestamp())
          .select("etapa", "perfil", "verificacao", "esperado", "obtido", "resultado", "executado_em"))
    df.write.mode("append").saveAsTable(VALIDACAO)

In [0]:
# O QUE FAZ: monta o plano de GRANT (lista de comandos) a partir do catálogo: lista os objetos da gold que
#   o consumidor pode ler, sem as tabelas de apoio das materialized views (__materialization_*) e sem o log do pipeline.
#
# FUNÇÕES DECLARADAS:
# - q(grupo): coloca o nome do grupo entre crases, como o SQL exige.
# - item(descricao, sql, alternativa): cria um item do plano; `alternativa` é um segundo comando tentado se o primeiro falhar.
#
# FUNÇÕES NATIVAS E SQL:
# - information_schema.tables: catálogo de tabelas e views. startswith(coluna, 'x'): verdadeiro se o nome começa com x.
# - GRANT USE CATALOG / USE SCHEMA / SELECT ... TO grupo: concede o privilégio (repetir não muda nada).
# - GRANT ALL PRIVILEGES ON CATALOG: todos os privilégios, herdados pelos schemas e objetos.
# - sorted(), set(), list comprehension: ordenam e montam as listas de objetos esperados.
def q(grupo: str) -> str:
    return f"`{grupo}`"


def item(descricao: str, sql: str, alternativa: str = None) -> tuple:
    return (descricao, sql, alternativa)


objetos_gold = sorted(r["table_name"] for r in spark.sql(f"""
    SELECT table_name FROM {CATALOG}.information_schema.tables
    WHERE table_schema = 'gold'
      AND NOT startswith(table_name, '__')
      AND NOT startswith(table_name, 'event_log_')
""").collect())

plano = []
for papel, grupo in GRUPOS.items():
    plano.append(item(f"{papel}: USE CATALOG", f"GRANT USE CATALOG ON CATALOG {CATALOG} TO {q(grupo)}"))

plano.append(item("engenharia: todos os privilégios", f"GRANT ALL PRIVILEGES ON CATALOG {CATALOG} TO {q(GRUPOS['engenharia'])}"))

for s in SCHEMAS_DPO:
    plano.append(item(f"dpo: leitura em {s}", f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOG}.{s} TO {q(GRUPOS['dpo'])}"))

for papel in CONSUMIDORES:
    g = q(GRUPOS[papel])
    plano.append(item(f"{papel}: USE SCHEMA gold", f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.gold TO {g}"))
    for o in objetos_gold:
        plano.append(item(f"{papel}: SELECT gold.{o}", f"GRANT SELECT ON TABLE {CATALOG}.gold.{o} TO {g}",
                          f"GRANT SELECT ON VIEW {CATALOG}.gold.{o} TO {g}"))
    plano.append(item(f"{papel}: USE SCHEMA quality", f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.quality TO {g}"))
    for v in VALIDACOES:
        plano.append(item(f"{papel}: SELECT quality.{v}", f"GRANT SELECT ON TABLE {CATALOG}.quality.{v} TO {g}"))

ESPERADO_CONSUMIDOR = {("gold", o) for o in objetos_gold} | {("quality", v) for v in VALIDACOES}
print(f"{len(objetos_gold)} objetos da gold liberados ao consumidor; {len(plano)} comandos no plano")
print("fora do plano (não concedidos):", sorted(r["table_name"] for r in spark.sql(
    f"SELECT table_name FROM {CATALOG}.information_schema.tables WHERE table_schema='gold' AND (startswith(table_name,'__') OR startswith(table_name,'event_log_'))").collect()))

In [0]:
# O QUE FAZ: no modo `simulacao` imprime cada comando; no modo `executar` aplica o comando (e a alternativa, se o
#   primeiro falhar) e junta as falhas. Para com erro no fim se algum comando não funcionou.
#
# FUNÇÕES DECLARADAS:
# - nenhuma (laço sobre o plano).
#
# FUNÇÕES NATIVAS E SQL:
# - spark.sql(texto): executa o comando SQL. try/except: tenta o comando e captura o erro sem parar o laço.
falhas = []
for descricao, sql, alternativa in plano:
    if MODO != "executar":
        print("[simulacao]", sql)
        continue
    try:
        spark.sql(sql)
        print("ok  ", descricao)
    except Exception as e1:
        if alternativa:
            try:
                spark.sql(alternativa)
                print("ok  ", descricao, "(pela alternativa)")
                continue
            except Exception as e2:
                e1 = e2
        falhas.append((descricao, str(e1)[:300]))
        print("FALHA", descricao, "->", str(e1)[:200])

if MODO == "executar" and falhas:
    raise RuntimeError(f"{len(falhas)} comando(s) falharam: {[f[0] for f in falhas]}")
print("\nplano impresso (simulacao)" if MODO != "executar" else "\ntodos os comandos aplicados")

In [0]:
# O QUE FAZ: lê os privilégios gravados no catálogo e compara com o plano. Grava as verificações em
#   quality.validacao_governanca e interrompe se alguma for reprovada. Só roda no modo `executar`.
#
# FUNÇÕES DECLARADAS:
# - escolher(df, candidatos): devolve o primeiro nome de coluna que existe no DataFrame (o nome muda entre views do
#   information_schema).
# - resumo(conjunto): texto curto com a contagem e até 4 itens.
#
# FUNÇÕES NATIVAS E SQL:
# - information_schema.table_privileges / schema_privileges / catalog_privileges: privilégios concedidos, um por linha.
# - df.where(F.col(c).isin(lista)): filtra as linhas cujo grantee está na lista. .collect(): traz as linhas ao driver
#   (volume pequeno: só os grupos do projeto).
# - conjunto - outro: itens que só estão no primeiro. set(): conjunto sem repetição.
def escolher(df, candidatos):
    for c in candidatos:
        if c in df.columns:
            return c
    raise RuntimeError(f"nenhuma das colunas {candidatos} em {df.columns}")


def resumo(conjunto) -> str:
    itens = sorted(".".join(x) if isinstance(x, tuple) else str(x) for x in conjunto)
    return f"{len(itens)}: {', '.join(itens[:4])}" + (" ..." if len(itens) > 4 else "")


if MODO != "executar":
    print("Verificação só roda no modo executar.")
else:
    nomes = list(GRUPOS.values())
    tp = spark.sql(f"SELECT * FROM {CATALOG}.information_schema.table_privileges")
    sp = spark.sql(f"SELECT * FROM {CATALOG}.information_schema.schema_privileges")
    cp = spark.sql(f"SELECT * FROM {CATALOG}.information_schema.catalog_privileges")
    t_g, t_s, t_t, t_p = escolher(tp, ["grantee"]), escolher(tp, ["table_schema", "schema_name"]), escolher(tp, ["table_name"]), escolher(tp, ["privilege_type"])
    s_g, s_s, s_p = escolher(sp, ["grantee"]), escolher(sp, ["schema_name", "table_schema"]), escolher(sp, ["privilege_type"])
    c_g, c_p = escolher(cp, ["grantee"]), escolher(cp, ["privilege_type"])

    priv_t = [(r[t_g], r[t_s], r[t_t], r[t_p]) for r in tp.where(F.col(t_g).isin(nomes)).collect()]
    priv_s = [(r[s_g], r[s_s], r[s_p]) for r in sp.where(F.col(s_g).isin(nomes)).collect()]
    priv_c = [(r[c_g], str(r[c_p]).replace("_", " ")) for r in cp.where(F.col(c_g).isin(nomes)).collect()]

    linhas = []
    for papel in CONSUMIDORES:
        g = GRUPOS[papel]
        obtido = {(s, t) for (gr, s, t, p) in priv_t if gr == g and p == "SELECT"}
        faltam, sobram = ESPERADO_CONSUMIDOR - obtido, obtido - ESPERADO_CONSUMIDOR
        linhas.append((papel, "SELECT somente nos objetos previstos (gold e validacao_*)", resumo(ESPERADO_CONSUMIDOR),
                       f"faltam {resumo(faltam)}; sobram {resumo(sobram)}" if (faltam or sobram) else resumo(obtido),
                       "OK" if not (faltam or sobram) else "REPROVADA"))
        fora = [(s, t) for (gr, s, t, p) in priv_t if gr == g and s in ("landing", "bronze", "silver", "ops")]
        fora += [(s, "*") for (gr, s, p) in priv_s if gr == g and s in ("landing", "bronze", "silver", "ops")]
        linhas.append((papel, "nenhum acesso a landing, bronze, silver e ops", "0", str(len(fora)), "OK" if not fora else "REPROVADA"))
        apoio = [(s, t) for (gr, s, t, p) in priv_t if gr == g and (t.startswith("__") or t.startswith("event_log_"))]
        linhas.append((papel, "sem acesso às tabelas de apoio e ao log do pipeline da gold", "0", str(len(apoio)), "OK" if not apoio else "REPROVADA"))

    g = GRUPOS["dpo"]
    tem = {s for (gr, s, p) in priv_s if gr == g and p == "SELECT"}
    linhas.append(("dpo", "SELECT em bronze, silver, gold, quality e ops", ", ".join(SCHEMAS_DPO), ", ".join(sorted(tem)),
                   "OK" if set(SCHEMAS_DPO) <= tem else "REPROVADA"))

    g = GRUPOS["engenharia"]
    prv = sorted({p for (gr, p) in priv_c if gr == g})
    linhas.append(("engenharia", "privilégios no catálogo olist", "ALL PRIVILEGES (ou todos os privilégios expandidos)", ", ".join(prv) or "nenhum",
                   "OK" if prv else "REPROVADA"))

    usos = {gr for (gr, p) in priv_c if p == "USE CATALOG"}
    linhas.append(("geral", "USE CATALOG para os 4 grupos", str(len(nomes)), str(len(usos & set(nomes))),
                   "OK" if set(nomes) <= usos else "REPROVADA"))

    registrar("6.2-grants", linhas)
    for l in linhas:
        print(f"[{l[4]:9}] {l[0]:10} {l[1]} | obtido: {l[3][:120]}")
    reprovadas = [l for l in linhas if l[4] == "REPROVADA"]
    if reprovadas:
        raise RuntimeError(f"{len(reprovadas)} verificação(ões) reprovada(s); veja a tabela {VALIDACAO}")
    print(f"\n{len(linhas)} verificações OK, gravadas em {VALIDACAO}")

## O que olhar no resultado

- No modo `simulacao`: a lista de comandos (cerca de 70) e a linha "fora do plano", que mostra o que **não** é concedido (tabelas `__materialization_*` e log do pipeline).
- No modo `executar`: cada linha `ok` e depois 12 verificações com `[OK]`. Se aparecer `REPROVADA`, a linha diz o que faltou ou sobrou. Consulte `SELECT * FROM olist.quality.validacao_governanca WHERE etapa = '6.2-grants'`.
- A verificação lê os privilégios gravados. Você, como dono e administrador, enxerga tudo por privilégio próprio; por isso a ausência de acesso à silver é provada pela lista de privilégios, e não por uma consulta sua.

**Próximo:** `02_classificacao_e_mascaras`.